# Live exercise: your data, your first number, your first run

You have a filled-in scoping worksheet
(`sessions/session-11/exercises/scoping-worksheet.md`). This notebook takes
it to a first working run before you leave today, in three parts:

1. **read your file** and give it the five-minute first look
2. **the one-number test**: a rough version of your must answer
3. **the first run**: the same thing inside `project/pipeline.py`

Ugly is the goal. A first run that prints one rough number from your own
data is worth more than a beautiful plan. Everything after today is
improving something that already works, which is a much nicer place to be.

Notebook 01 did all of this on the music data, so when you are stuck, look
at how it did the same step.

In [ ]:
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

## Part 1: read your file

Copy your data file into `project/data/`. Copy it, do not move it: keep the
original somewhere safe, so nothing you do here can damage it.

Then put its path below, counted from the repository root, like every
other path in the course.

In [ ]:
import pandas as pd

DATA_PATH = Path("project/data/YOUR-FILE-NAME.csv")      # TODO: your file

print(DATA_PATH, "exists:", DATA_PATH.exists())

If that says `False`, check the spelling and the extension before anything
else. Windows hides extensions by default, so `export.csv` can really be
`export.csv.txt`. `print(list(Path("project/data").iterdir()))` shows what
is actually there.

### Three things that go wrong with files from a spreadsheet

Run this cell. It is a tiny file, made up on the spot, in the shape Dutch
Excel saves a CSV: semicolons between columns and a comma in the decimals.

In [ ]:
import io

dutch_csv = io.StringIO(
    "datum;categorie;bedrag\n"
    "2025-01-03;boodschappen;54,20\n"
    "2025-01-04;vervoer;12,85\n"
)

wrong = pd.read_csv(dutch_csv)
print(wrong.columns.tolist())
print(len(wrong.columns), "column")

No error, and one column called `datum;categorie;bedrag`. pandas expected
commas, found none, and decided the whole line was one value. Telling it
what the file really looks like fixes both problems at once:

In [ ]:
dutch_csv.seek(0)                    # rewind the pretend file to the start
right = pd.read_csv(dutch_csv, sep=";", decimal=",")
print(right)
print(right.dtypes)

The three to know:

| Symptom | Cause | Fix |
|---|---|---|
| one column with semicolons in its name | the file uses `;` between columns | `sep=";"` |
| numbers are `object`, like `"54,20"` | decimal comma | `decimal=","` |
| `UnicodeDecodeError` on the first accented letter | saved in the old Windows encoding | `encoding="cp1252"` |

**An `.xlsx` file** can be read directly with `pd.read_excel(path,
sheet_name="...")`, but it needs an extra package, `openpyxl`, which is not
in `project/requirements.txt`. For today the simplest route is *Save As ->
CSV* in Excel. If the project really needs the Excel file, install
`openpyxl` into the project's virtual environment and add it to
`requirements.txt`, so the next person (or you, next month) gets it too.

Now read your own file. Add whichever of `sep`, `decimal` and `encoding`
it needs.

In [ ]:
# TODO: raw = pd.read_csv(DATA_PATH, ...)

### The five-minute first look

This is the function from notebook 01, unchanged. Run it on your data.
Pass the name of your date column if you have one.

In [ ]:
def first_look(df, date_column=None):
    """Print the four things to check on any new dataset."""
    print(f"rows: {len(df):,}   columns: {len(df.columns)}")
    print()
    print("types and missing values per column:")
    summary = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum()})
    print(summary)
    if date_column is not None:
        print()
        print(f"{date_column}: {df[date_column].min()} to {df[date_column].max()}")


# TODO: first_look(raw, date_column="...")

Write down what you see, in the *What the data looks like* part of your
worksheet. In particular:

* is any column you need the wrong type? (numbers as `object` usually means
  a stray character, a decimal comma, or a total row at the bottom)
* is anything missing in a column your question uses?
* does the date range cover what your question needs?

If the answer to any of these changes your question, change the worksheet
now. That is not failure; it is the first look doing its job.

*Your notes here.*

## Part 2: the one-number test

Copy your question and your **must** output from the worksheet here.

**My question:** ...

**My must output:** ...

Now get a rough version of it. Not the finished output: the one number or
the small table at its heart. It will usually be one of the shapes from
session 8: a filter, a `value_counts`, or a `groupby` with `sum` or
`mean`.

Fifteen lines is a good target. If it is turning into fifty, stop and talk
about it: the question probably needs cutting down.

In [ ]:
# TODO: the rough answer

### Can you believe it?

Write one `assert` that checks your number against something you know
independently. Ideas:

* the parts add up to the whole (notebook 01 checked the genre table added
  up to the raw total)
* the row count is what the spreadsheet showed
* one value you can look up by hand matches

A number with no check is a number you are hoping about.

In [ ]:
# TODO: one assert

Say the answer as a sentence, the way you would in session 12. An f-string
with `:.1f` (or `:+.1f`, or `:,`) is the right tool.

In [ ]:
# TODO: print the answer as a sentence

## Part 3: the first run of `project/pipeline.py`

Now the same thing, inside the pipeline from session 10. This happens in
VS Code and the terminal, not in this notebook. Work through it in order,
and run the pipeline after **every** step, so that when something breaks
you know which change broke it.

1. **Run it unchanged first**, from the repository root, with the project's
   virtual environment active:

   ```
   python project/pipeline.py
   ```

   It should still write `summary.csv` and `chart.png` into
   `project/output/`. If it does not, fix that before changing anything.
   You need a known-good starting point.
2. **Point it at your file.** Every place to change is marked
   `MAKE IT YOURS`. Near the top, set `INPUT` to your file, for example
   `INPUT = PROJECT / "data" / "my_export.csv"`, and `QUESTION` to your
   question. Give `load` the same `read_csv` arguments you needed above.
3. **Cut `clean` down to almost nothing.** The music version renames and
   fixes columns your file does not have, so it will fail. Replace its body
   with: keep the columns your question needs, plus the one fix that stops
   the analysis running. Real cleaning is milestone 1, this week.
4. **Replace the body of `analyse`** with your rough answer from part 2,
   returning a table instead of displaying it. End it with
   `.reset_index()`: `save_csv` writes without the index, so a table whose
   labels live in the index would lose them.
5. **The chart and the check.** `save_chart` expects a `genre` and a
   `minutes` column, and `check` expects a `plays` column, so both will
   complain. Change the column names if your table has an obvious pair;
   otherwise comment out the two calls in `main()` with a
   `# TODO (milestone 1)` and `# TODO (milestone 2)`, so you do not forget.
6. **Run it.** Open `project/output/summary.csv`. Does it hold the same
   number you got in part 2?

Exploring that does not fit in this notebook belongs in
`project/notebooks/explore.ipynb`: the project's own place for trying
things before they move into `pipeline.py`.

Then commit. Not the data (see the slide about private data first), but
the changed `pipeline.py`:

```
git add project/pipeline.py
git commit -m "First run of the project pipeline on my own data"
```

When the CSV in `project/output/` holds your number, today is done.